In [ ]:
%load_ext autoreload
%autoreload 2

# NF density near the pole and comparison with the simulation

Component 01. Why the polar pixels 0, 3, 3068, 3071 converge slowly in notebook 09: the density of the flow
near the pole, and the simulated events near χ = 0 and the pole compared with the NF samples.

## 1. Setup

In [ ]:
import sys
from pathlib import Path
import h5py
import numpy as np
import healpy as hp
from astropy.io import fits
from histpy import Histogram
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))     # project root, so `src` is importable
from src.nf_binned_template import NFBinnedTemplate   # imports cosipy

from src.plot_style import COLORS, set_global_style
set_global_style()                             # after cosipy

%matplotlib inline

In [ ]:
# NF model (component 01), binning of the notebook 08 histograms, simulated events and NF samples
MODEL_PATH = Path("../model/nfbackground_part-01_v3-00.fits")
AXES = Histogram.open("../data/NF_background/part-01_full_caffeinated_511keV_DC4binning_local.hdf5").axes
NSIDE = 16
SIM_PATH = Path("../data/background_components/CosmicPhotons_3months_unbinned_data_filtered_with_SAAcut.fits")
NF_SAMPLES_PATH = Path("../data/NF_background/part-01_full_caffeinated.hdf5")

# Cheap integration settings (as in notebook 09): 2 Em points on each side of 511 keV, 1 Phi point per bin
QUAD = dict(n_em_per_bin=2, n_phi_per_bin=1, em_breaks_kev=[511.0])

# [script] Template without the chi = 0 correction: the raw density of the flow
template = NFBinnedTemplate(MODEL_PATH, seam_delta_deg=0)

### 1.1 Density tool

Density along lines on the sky, in counts per pixel area (as in notebook 09, Section 8).

In [ ]:
# Em/Phi points at QUAD: the weights below are those of the last compute
template.compute(AXES, [1440], subpixel_level=0, verbose=False, **QUAD)

# Sky density of the NF, integrated over the Em bin and all Phi, converted to counts per pixel area
# (same units as the table: model_counts x density [1/sr] x pixel area [sr])
w_em = template._em_matrix.sum(axis=0)       # [script] weight of each Em point 
w_phi = template._phi_matrix.sum(axis=0)     # [script] weight of each Phi point
pix_area = hp.nside2pixarea(NSIDE)           # [sr]

def counts_per_pixel_area(colat_deg, chi_deg):
    # [script] NF density at (Em points) x (Phi points) x (sky points), then the weighted sum over Em and Phi
    p = template._density_on_grid(np.deg2rad(colat_deg), np.deg2rad(np.mod(chi_deg, 360)))
    return template.model_counts * np.einsum("e,p,eps->s", w_em, w_phi, p) * pix_area

## 2. Density over the full χ range near the pole

In [ ]:
# Density over the full chi range at small colatitudes: does it become independent of chi toward the pole?
chi = np.linspace(0, 360, 721)                       # 0.5 deg steps
for colat in (0.5, 1, 2, 5):
    plt.plot(chi, counts_per_pixel_area(np.full_like(chi, colat), chi), label=rf"colatitude {colat}$^\circ$")
plt.yscale("log"); plt.xlabel(r"$\chi$ [deg]"); plt.ylabel("counts per pixel area"); plt.legend(); plt.show()

## 3. Simulation vs NF samples (all energies)

Same lines, with events instead of density (97.6 M events per file, north pole only).
Reads both files once (several minutes); the counts for Section 4 are filled in the same pass.

In [ ]:
# Simulation and NF samples, ALL energies, one pass over each file (Psi local, Chi local [rad]):
# 1. chi across the chi = 0 seam, colatitude 85-90 deg (around pixel 1440), 0.05 deg bins       -> Section 3
# 2. chi around the ring near the north pole, colatitude 0-1, 1-2, 2-5 deg, 5 deg bins          -> Section 3
# 3. events within TOL of chi = 0, 90, 180, 270 deg and of psi = 0, 180 deg                    -> Section 4
seam_edges = np.arange(-2, 2.0001, 0.05)                       # deg; chi > 180 deg is shown as chi - 360
seam_band = (85.0, 90.0)                                       # colatitude [deg]
ring_edges = np.arange(0, 360.0001, 5.0)                       # deg
pole_bands = [(0.0, 1.0), (1.0, 2.0), (2.0, 5.0)]              # colatitude [deg]
TOL = 1e-6                                                     # rad (0.00006 deg)
quarters = {0: 0.0, 90: np.pi / 2, 180: np.pi, 270: 3 * np.pi / 2}   # rad

def fill(acc, psi_rad, chi_rad):
    psi_rad, chi_rad = np.asarray(psi_rad, dtype=np.float64), np.asarray(chi_rad, dtype=np.float64)
    colat, chi = np.rad2deg(psi_rad), np.rad2deg(chi_rad)
    in_band = (colat >= seam_band[0]) & (colat < seam_band[1])
    acc["seam"] += np.histogram(np.where(chi > 180, chi - 360, chi)[in_band], bins=seam_edges)[0]
    acc["ring"] += np.array([np.histogram(chi[(colat >= lo) & (colat < hi)], bins=ring_edges)[0] for lo, hi in pole_bands])
    dist = lambda q: np.abs((chi_rad - q + np.pi) % (2 * np.pi) - np.pi)   # angular distance of chi to q
    exact = {f"chi = {deg:3d} deg": np.sum(dist(q) < TOL) for deg, q in quarters.items()}
    exact["psi =   0 deg"] = np.sum(psi_rad < TOL)
    exact["psi = 180 deg"] = np.sum(psi_rad > np.pi - TOL)
    exact["psi = 0 deg and chi = 0 deg"] = np.sum((psi_rad < TOL) & (dist(0.0) < TOL))
    for key, value in exact.items():
        acc["exact"][key] = acc["exact"].get(key, 0) + int(value)

CHUNK = 2_000_000
acc = {name: {"seam": np.zeros(len(seam_edges) - 1), "ring": np.zeros((len(pole_bands), len(ring_edges) - 1)), "exact": {}}
       for name in ("sim", "nf")}

with fits.open(SIM_PATH, memmap=True) as hdul:                 # simulation (10.9 GB): only two columns per chunk
    ev = hdul[1].data
    for a in range(0, len(ev), CHUNK):
        fill(acc["sim"], ev["Psi local"][a:a + CHUNK], ev["Chi local"][a:a + CHUNK])

with h5py.File(NF_SAMPLES_PATH, "r") as f:                   # NF samples
    for a in range(0, f["Psi local"].shape[0], CHUNK):
        fill(acc["nf"], f["Psi local"][a:a + CHUNK], f["Chi local"][a:a + CHUNK])

print(f"seam band: simulation {acc['sim']['seam'].sum():,.0f}, NF samples {acc['nf']['seam'].sum():,.0f} events")
for (lo, hi), s, n in zip(pole_bands, acc["sim"]["ring"], acc["nf"]["ring"]):
    print(f"colatitude {lo:.0f}-{hi:.0f} deg: simulation {s.sum():,.0f}, NF samples {n.sum():,.0f} events")

In [ ]:
# chi across the chi = 0 seam (colatitude 85-90 deg): a dip at 0 in the NF samples but not in the simulation
# means the dip is an artifact of the flow (the fix replaces it by the neighbouring density)
seam_centres = 0.5 * (seam_edges[:-1] + seam_edges[1:])
plt.errorbar(seam_centres, acc["sim"]["seam"], yerr=np.sqrt(acc["sim"]["seam"]), fmt=".", color=COLORS["sim"], label="simulation")
plt.step(seam_centres, acc["nf"]["seam"], where="mid", color=COLORS["nf_samples"], label="NF samples")
plt.axvline(0, color=COLORS["ref"], ls="--", label=r"$\chi$ = 0 seam")
plt.xlabel(r"$\chi$ [deg]"); plt.ylabel(r"events per 0.05$^\circ$ (all energies)")
plt.title(r"colatitude 85-90$^\circ$"); plt.legend(); plt.show()

In [ ]:
# chi around the ring near the north pole: is the peak at chi ~ 0 (Section 2) also in the simulation?
ring_centres = 0.5 * (ring_edges[:-1] + ring_edges[1:])
fig, axs = plt.subplots(1, len(pole_bands), figsize=(17, 4.8))
for ax, (lo, hi), s, n in zip(axs, pole_bands, acc["sim"]["ring"], acc["nf"]["ring"]):
    ax.errorbar(ring_centres, s, yerr=np.sqrt(s), fmt=".", color=COLORS["sim"], label="simulation")
    ax.step(ring_centres, n, where="mid", color=COLORS["nf_samples"], label="NF samples")
    ax.set(title=rf"colatitude {lo:.0f}-{hi:.0f}$^\circ$", xlabel=r"$\chi$ [deg]", ylabel=r"events per 5$^\circ$ (all energies)")
    ax.legend()
plt.show()

## 4. Exact angles in the simulation

Events within 10⁻⁶ rad of χ = 0, 90°, 180°, 270° and ψ = 0, 180° (a smooth distribution gives ~15–30).
Counted in the Section 3 pass.

In [ ]:
# Events within TOL of chi = 0, 90, 180, 270 deg and of psi = 0, 180 deg (filled in the Section 3 pass)
print(f"{'':30s}{'simulation':>14s}{'NF samples':>14s}")
for key in acc["sim"]["exact"]:
    print(f"{key:30s}{acc['sim']['exact'][key]:14,d}{acc['nf']['exact'][key]:14,d}")